# Federal Reserve Macro Insights MVP (Streamlined)

This notebook is the clean orchestrator version.
Core logic is now in `core/*.py` modules.

Original notebook preserved: `fed_macro_mvp.ipynb`.


## 0) Install / refresh dependencies

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## 1) Imports and configuration

In [2]:
from pathlib import Path
import pandas as pd

from core.pipeline import create_config, run_ingest_and_index, run_full_analysis, persist_results

cfg = create_config(Path.cwd())
print('Project dir:', cfg.project_dir)
print('Model:', cfg.ollama_model)


/Users/atheeshkrishnan/AK/DEV/hawkdove/storied/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project dir: /Users/atheeshkrishnan/AK/DEV/hawkdove/fed_macro_mvp
Model: llama3:8b


## 2) Optional runtime overrides

In [3]:
# Recommended baseline on local CPU
cfg.top_k_topic = 2
cfg.max_context_chars = 2400
cfg.ollama_num_predict = 380
cfg.enable_reranker = False

# Keep retries + robust parser enabled
cfg.ollama_max_retries = 3
cfg.retry_context_shrink = [1.0, 0.85, 0.7]
cfg.retry_predict_shrink = [1.0, 1.0, 0.9]
cfg.use_json_repair = True

print('Overrides applied.')

Overrides applied.


## 3) Ingestion + indexing

In [4]:
ingest_index_result = run_ingest_and_index(cfg)

catalog_df = ingest_index_result['catalog_df']
download_df = ingest_index_result['download_df']
chunks_df = ingest_index_result['chunks_df']

print('Catalog candidates:', len(catalog_df))
print('Downloaded/available:', len(download_df))
print('Chunks:', len(chunks_df))

if not download_df.empty:
    display(download_df[['title', 'status', 'local_path']].head(10))

Batches: 100%|██████████| 49/49 [00:29<00:00,  1.68it/s]

Catalog candidates: 161
Downloaded/available: 40
Chunks: 1551


,title,status,local_path
0,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
1,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
2,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
3,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
4,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
5,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
6,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
7,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
8,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
9,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...


## 4) Macro analysis

In [5]:
analysis_result = run_full_analysis(cfg)

print('Sparse enabled:', analysis_result['sparse_enabled'])
print('Reranker enabled:', analysis_result['reranker_enabled'])
print('Timings:', analysis_result['timings'])

display(analysis_result['topic_summary_df'])
if not analysis_result['hits_df'].empty:
    display(analysis_result['hits_df'][['topic', 'chunk_id', 'doc_id', 'final_score', 'recency']].head(12))
if not analysis_result['attempt_log'].empty:
    display(analysis_result['attempt_log'])

if 'citation_preview_df' in analysis_result and not analysis_result['citation_preview_df'].empty:
    display(analysis_result['citation_preview_df'][['doc_id', 'chunk_id', 'quote']].head(8))

print(analysis_result.get('normalized_json_text', analysis_result['llm_text']))
if analysis_result['parsed'] is None:
    print('[check] JSON parse failed after retries')
else:
    quality = analysis_result['quality']
    print('[check] Missing top keys:', quality['missing_top_keys'] if quality['missing_top_keys'] else 'None')
    print('[check] Bad shape:', quality['bad_shape'] if quality['bad_shape'] else 'None')
    print('[check] Bad evidence IDs:', quality['bad_evidence_ids'] if quality['bad_evidence_ids'] else 'None')
    print('[check] Unknown citation IDs:', quality['unknown_citation_ids'] if quality['unknown_citation_ids'] else 'None')
    print('[check] Missing topics:', quality['missing_topics'] if quality['missing_topics'] else 'None')
    print('[check] Extra topics:', quality['extra_topics'] if quality['extra_topics'] else 'None')

Sparse enabled: True
Reranker enabled: False
Timings: {'sparse_latency_s': 0.2958638668060303, 'rerank_load_s': 1.8835067749023438e-05, 'retrieval_latency_s': 0.7331514358520508, 'llm_stage_s': 135.8158097267151}


,topic,hits,top_final_score,top_chunk_id
0,inflation,2,0.702196,fomcminutes20241107.pdf::chunk0011
1,unemployment,2,0.824892,fomcminutes20250917.pdf::chunk0020
2,growth,2,0.724387,20250207_mprfullreport.pdf::chunk0175
3,policy_rates,2,0.724387,20250207_mprfullreport.pdf::chunk0175
4,financial_conditions,2,0.693055,fomcminutes20240918.pdf::chunk0000
5,credit,2,0.702196,fomcminutes20241107.pdf::chunk0016


,topic,chunk_id,doc_id,final_score,recency
0,unemployment,fomcminutes20250917.pdf::chunk0020,fomcminutes20250917.pdf,0.824892,0.499691
1,growth,20250207_mprfullreport.pdf::chunk0175,20250207_mprfullreport.pdf,0.724387,0.212535
2,inflation,fomcminutes20241107.pdf::chunk0011,fomcminutes20241107.pdf,0.702196,0.149132
3,credit,fomcminutes20241107.pdf::chunk0016,fomcminutes20241107.pdf,0.702196,0.149132
4,financial_conditions,fomcminutes20240918.pdf::chunk0000,fomcminutes20240918.pdf,0.693055,0.123013
5,financial_conditions,20250620_mprfullreport.pdf::chunk0013,20250620_mprfullreport.pdf,0.558398,0.354698
6,credit,fomcminutes20260128.pdf::chunk0016,fomcminutes20260128.pdf,0.191892,0.833929
7,growth,fomcprojtabl20250618.pdf::chunk0001,fomcprojtabl20250618.pdf,0.173428,0.351977
8,policy_rates,fomcminutes20250507.pdf::chunk0008,fomcminutes20250507.pdf,0.150517,0.299415
9,unemployment,fomcminutes20250319.pdf::chunk0000,fomcminutes20250319.pdf,0.146126,0.247928


,attempt,status,latency_s,context_chars,num_predict
0,1,ok,135.76,2400,380


,doc_id,chunk_id,quote
0,fomcminutes20241107.pdf,fomcminutes20241107.pdf::chunk0011,Financial Situation The expected path of the f...
1,20250207_mprfullreport.pdf,20250207_mprfullreport.pdf::chunk0175,.” Figure 47. Selected interest rates Departme...
2,fomcminutes20250917.pdf,fomcminutes20250917.pdf::chunk0020,he unemployment rate had edged up. Participant...
3,fomcminutes20250319.pdf,fomcminutes20250319.pdf::chunk0000,FOMC FEDERAL RESERVE SYSTEM Minutes of the Fed...
4,fomcprojtabl20250618.pdf,fomcprojtabl20250618.pdf::chunk0001,most likely to foster outcomes for economic ac...
5,fomcminutes20250507.pdf,fomcminutes20250507.pdf::chunk0008,lable data pointed to modest outflows from fix...
6,fomcminutes20240918.pdf,fomcminutes20240918.pdf::chunk0000,1 Minutes of the Federal Open Market Committee...
7,20250620_mprfullreport.pdf,20250620_mprfullreport.pdf::chunk0013,"well as manufacturing output, rose solidly in ..."


{
  "generated_at_utc": "2026-03-16T03:53:42Z",
  "executive_summary": "Recent Federal Reserve communications suggest a nuanced macro view for the next 6-12 months.",
  "regime_call": {
    "growth_momentum": "Moderate",
    "inflation_trend": "Tame",
    "policy_bias": "Neutral",
    "recession_risk": "Low",
    "confidence": "0.7"
  },
  "topic_signals": [
    {
      "topic": "inflation",
      "view": "Stable",
      "confidence": "0.7",
      "evidence": [
        "fomcminutes20241107.pdf::chunk0011",
        "20250207_mprfullreport.pdf::chunk0175"
      ]
    },
    {
      "topic": "unemployment",
      "view": "Stable",
      "confidence": "0.8",
      "evidence": [
        "fomcminutes20250917.pdf::chunk0020",
        "fomcminutes20250319.pdf::chunk0000"
      ]
    },
    {
      "topic": "growth",
      "view": "Moderate",
      "confidence": "0.7",
      "evidence": [
        "20250207_mprfullreport.pdf::chunk0175",
        "fomcprojtabl20250618.pdf::chunk0001"
      ]
    

## 5) Save artifacts

In [6]:
saved_paths = persist_results(cfg, analysis_result)
print(saved_paths)

{'hits_path': PosixPath('/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_macro_mvp/outputs/hits_20260315_235342.csv'), 'text_path': PosixPath('/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_macro_mvp/outputs/macro_answer_20260315_235342.txt'), 'json_path': PosixPath('/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_macro_mvp/outputs/macro_answer_20260315_235342.json')}
